# Análise do Mercado de Ações

Análise exploratória de cinco empresas (IBM, Microsoft, Tesla, Oracle e Walmart) entre fev/2022 e fev/2023.

Colunas do arquivo: Empresa, Data, Close, Volume, Open, High, Low.

Seções:
1. Carga e limpeza dos dados
2. Visão geral e estatísticas descritivas
3. Evolução de preços
4. Retornos diários e risco
5. Correlação entre as ações
6. Médias móveis (empresa à escolha)
7. Volume negociado
8. Resumo comparativo

## 1. Configuração e carga dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
pd.options.display.float_format = "{:,.2f}".format

# Caminho do arquivo (deixe o .xlsx na mesma pasta do notebook ou ajuste aqui)
ARQUIVO = "StockMarket.xlsx"

# Empresa usada na seção de médias móveis
EMPRESA_FOCO = "Tesla"

In [ ]:
df = pd.read_excel(ARQUIVO)

# A coluna Data vem como texto no formato mm/dd/aaaa
df["Data"] = pd.to_datetime(df["Data"], format="%m/%d/%Y")
df = df.sort_values(["Empresa", "Data"]).reset_index(drop=True)

df.head()

## 2. Visão geral e qualidade dos dados

In [ ]:
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print(f"Período: {df['Data'].min():%d/%m/%Y} até {df['Data'].max():%d/%m/%Y}")
print(f"Valores nulos: {df.isna().sum().sum()}")
print(f"Linhas duplicadas (Empresa + Data): {df.duplicated(['Empresa', 'Data']).sum()}")
print()
df.groupby("Empresa").agg(
    pregoes=("Data", "count"),
    inicio=("Data", "min"),
    fim=("Data", "max"),
)

In [ ]:
# Consistência dos preços: High deve ser o maior valor do dia e Low o menor
inconsistentes = df[
    (df["High"] < df[["Open", "Close"]].max(axis=1) - 1e-6)
    | (df["Low"] > df[["Open", "Close"]].min(axis=1) + 1e-6)
]
print(f"Linhas com High/Low inconsistentes: {len(inconsistentes)}")
inconsistentes.head()

### Estatísticas descritivas por empresa

In [ ]:
df.groupby("Empresa")[["Close", "Volume"]].describe().T

## 3. Evolução de preços

In [ ]:
# Preço de fechamento (escala real)
fig, ax = plt.subplots()
for empresa, g in df.groupby("Empresa"):
    ax.plot(g["Data"], g["Close"], label=empresa, linewidth=1.5)
ax.set_title("Preço de fechamento por empresa")
ax.set_ylabel("Close (US$)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.legend()
plt.show()

In [ ]:
# Preço normalizado (base 100 no primeiro pregão) para comparar desempenho relativo
pivot_close = df.pivot(index="Data", columns="Empresa", values="Close").sort_index()
base100 = pivot_close / pivot_close.iloc[0] * 100

fig, ax = plt.subplots()
base100.plot(ax=ax, linewidth=1.5)
ax.axhline(100, color="gray", linestyle="--", linewidth=1)
ax.set_title("Desempenho relativo (base 100 no início do período)")
ax.set_ylabel("Índice")
ax.legend(title="")
plt.show()

print("Variação total no período (%):")
((base100.iloc[-1] - 100)).sort_values(ascending=False).to_frame("Retorno (%)")

In [ ]:
# Um gráfico por empresa, mostrando a faixa diária (Low-High)
empresas = sorted(df["Empresa"].unique())
fig, axes = plt.subplots(len(empresas), 1, figsize=(12, 3 * len(empresas)), sharex=True)

for ax, empresa in zip(axes, empresas):
    g = df[df["Empresa"] == empresa]
    ax.fill_between(g["Data"], g["Low"], g["High"], alpha=0.3, label="Faixa Low-High")
    ax.plot(g["Data"], g["Close"], linewidth=1.2, label="Close")
    ax.set_title(empresa)
    ax.set_ylabel("US$")
    ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

## 4. Retornos diários e risco

In [ ]:
pivot_close = df.pivot(index="Data", columns="Empresa", values="Close").sort_index()
retornos = pivot_close.pct_change().dropna()

DIAS_UTEIS_ANO = 252

resumo_risco = pd.DataFrame({
    "Retorno diário médio (%)": retornos.mean() * 100,
    "Volatilidade diária (%)": retornos.std() * 100,
    "Volatilidade anualizada (%)": retornos.std() * np.sqrt(DIAS_UTEIS_ANO) * 100,
    "Melhor dia (%)": retornos.max() * 100,
    "Pior dia (%)": retornos.min() * 100,
    "Retorno acumulado (%)": ((1 + retornos).prod() - 1) * 100,
})

# Drawdown máximo: maior queda desde um pico anterior
cummax = pivot_close.cummax()
drawdown = pivot_close / cummax - 1
resumo_risco["Drawdown máximo (%)"] = drawdown.min() * 100

# Relação retorno/risco simplificada (sem taxa livre de risco)
resumo_risco["Retorno/Risco"] = (
    retornos.mean() * DIAS_UTEIS_ANO / (retornos.std() * np.sqrt(DIAS_UTEIS_ANO))
)

resumo_risco.sort_values("Retorno acumulado (%)", ascending=False)

In [ ]:
# Distribuição dos retornos diários
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=retornos * 100, ax=axes[0])
axes[0].set_title("Retornos diários por empresa (%)")
axes[0].set_ylabel("%")
axes[0].set_xlabel("")

for col in retornos.columns:
    sns.kdeplot(retornos[col] * 100, ax=axes[1], label=col, fill=True, alpha=0.15)
axes[1].set_title("Densidade dos retornos diários (%)")
axes[1].set_xlabel("%")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Drawdown ao longo do tempo
fig, ax = plt.subplots()
(drawdown * 100).plot(ax=ax, linewidth=1.3)
ax.set_title("Drawdown (queda desde o pico anterior)")
ax.set_ylabel("%")
ax.legend(title="")
plt.show()

In [ ]:
# Volatilidade móvel (janela de 21 pregões, anualizada)
vol_movel = retornos.rolling(21).std() * np.sqrt(DIAS_UTEIS_ANO) * 100

fig, ax = plt.subplots()
vol_movel.plot(ax=ax, linewidth=1.3)
ax.set_title("Volatilidade móvel de 21 pregões (anualizada)")
ax.set_ylabel("%")
ax.legend(title="")
plt.show()

### Retorno mensal

In [ ]:
# Retorno por mês (último fechamento do mês vs. último do mês anterior)
try:
    fechamento_mensal = pivot_close.resample("ME").last()   # pandas >= 2.2
except ValueError:
    fechamento_mensal = pivot_close.resample("M").last()    # pandas mais antigo
mensal = fechamento_mensal.pct_change().dropna() * 100
mensal.index = mensal.index.strftime("%Y-%m")

plt.figure(figsize=(8, 6))
sns.heatmap(mensal, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            cbar_kws={"label": "%"})
plt.title("Retorno mensal (%)")
plt.ylabel("")
plt.xlabel("")
plt.show()

## 5. Correlação entre as ações

In [ ]:
corr = retornos.corr()

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlação dos retornos diários")
plt.show()

In [ ]:
sns.pairplot(retornos * 100, corner=True, plot_kws={"alpha": 0.4, "s": 15})
plt.suptitle("Dispersão entre retornos diários (%)", y=1.02)
plt.show()

## 6. Médias móveis

Altere a variável `EMPRESA_FOCO` na primeira célula para analisar outra empresa.

In [ ]:
g = df[df["Empresa"] == EMPRESA_FOCO].set_index("Data").sort_index().copy()
g["MM20"] = g["Close"].rolling(20).mean()
g["MM50"] = g["Close"].rolling(50).mean()

# Cruzamentos entre MM20 e MM50
sinal = np.sign(g["MM20"] - g["MM50"])
cruz = g[(sinal != sinal.shift()) & sinal.notna() & sinal.shift().notna()]
altas = cruz[cruz["MM20"] > cruz["MM50"]]
baixas = cruz[cruz["MM20"] < cruz["MM50"]]

fig, ax = plt.subplots()
ax.plot(g.index, g["Close"], label="Close", linewidth=1, alpha=0.7)
ax.plot(g.index, g["MM20"], label="MM20", linewidth=1.6)
ax.plot(g.index, g["MM50"], label="MM50", linewidth=1.6)
ax.scatter(altas.index, altas["Close"], marker="^", color="green", s=80, label="Cruzamento de alta", zorder=5)
ax.scatter(baixas.index, baixas["Close"], marker="v", color="red", s=80, label="Cruzamento de baixa", zorder=5)
ax.set_title(f"{EMPRESA_FOCO}: preço e médias móveis")
ax.set_ylabel("US$")
ax.legend()
plt.show()

In [ ]:
# Bandas de Bollinger (20 pregões, 2 desvios)
std20 = g["Close"].rolling(20).std()
g["BB_sup"] = g["MM20"] + 2 * std20
g["BB_inf"] = g["MM20"] - 2 * std20

fig, ax = plt.subplots()
ax.plot(g.index, g["Close"], label="Close", linewidth=1.2)
ax.plot(g.index, g["MM20"], label="MM20", linewidth=1, color="gray")
ax.fill_between(g.index, g["BB_inf"], g["BB_sup"], alpha=0.2, label="Bandas de Bollinger")
ax.set_title(f"{EMPRESA_FOCO}: Bandas de Bollinger")
ax.set_ylabel("US$")
ax.legend()
plt.show()

## 7. Volume negociado

In [ ]:
vol = df.pivot(index="Data", columns="Empresa", values="Volume").sort_index()

print("Volume médio diário:")
display(vol.mean().sort_values(ascending=False).to_frame("Volume médio").style.format("{:,.0f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

(vol.rolling(21).mean() / 1e6).plot(ax=axes[0], linewidth=1.3)
axes[0].set_title("Volume médio móvel de 21 pregões (milhões)")
axes[0].set_ylabel("Milhões de ações")
axes[0].legend(title="")

sns.boxplot(data=vol / 1e6, ax=axes[1])
axes[1].set_title("Distribuição do volume diário (milhões)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Volume costuma subir em dias de grande variação? Correlação entre |retorno| e volume
abs_ret = retornos.abs()
corr_vol = pd.Series({
    emp: abs_ret[emp].corr(vol[emp].loc[abs_ret.index]) for emp in abs_ret.columns
}, name="Correlação |retorno| x volume")
corr_vol.to_frame()

## 8. Resumo comparativo

In [ ]:
# Amplitude intradiária média: (High - Low) / Open
df["Amplitude (%)"] = (df["High"] - df["Low"]) / df["Open"] * 100
amplitude = df.groupby("Empresa")["Amplitude (%)"].mean()

resumo = resumo_risco[[
    "Retorno acumulado (%)", "Volatilidade anualizada (%)",
    "Drawdown máximo (%)", "Retorno/Risco",
]].copy()
resumo["Amplitude intradiária média (%)"] = amplitude
resumo["Volume médio"] = vol.mean()

(resumo.sort_values("Retorno acumulado (%)", ascending=False)
       .style.format({"Volume médio": "{:,.0f}"}, precision=2)
       .background_gradient(cmap="RdYlGn", subset=["Retorno acumulado (%)", "Retorno/Risco"])
       .background_gradient(cmap="RdYlGn_r", subset=["Volatilidade anualizada (%)"]))

In [ ]:
melhor = resumo["Retorno acumulado (%)"].idxmax()
pior = resumo["Retorno acumulado (%)"].idxmin()
mais_volatil = resumo["Volatilidade anualizada (%)"].idxmax()
menos_volatil = resumo["Volatilidade anualizada (%)"].idxmin()

print(f"Maior retorno no período:   {melhor} ({resumo.loc[melhor, 'Retorno acumulado (%)']:.1f}%)")
print(f"Menor retorno no período:   {pior} ({resumo.loc[pior, 'Retorno acumulado (%)']:.1f}%)")
print(f"Ação mais volátil:          {mais_volatil} ({resumo.loc[mais_volatil, 'Volatilidade anualizada (%)']:.1f}% a.a.)")
print(f"Ação menos volátil:         {menos_volatil} ({resumo.loc[menos_volatil, 'Volatilidade anualizada (%)']:.1f}% a.a.)")